# DSEG660 GRPO Lab — Colab runner
Runtime → Change runtime type → **T4 GPU**. Run cells top to bottom. Each stage runs as its own process so GPU memory is released between stages.

In [ ]:
!pip install -q unsloth
!nvidia-smi

Upload `dseg660-grpo-lab.zip` (this folder) and `dseg660-cod-data.zip` with the Files pane (or `files.upload()` below), then unzip.

In [ ]:
from google.colab import files
import os
if not os.path.exists('dseg660-cod-data.zip'): files.upload()
!unzip -oq dseg660-grpo-lab.zip -d lab && unzip -oq dseg660-cod-data.zip -d lab/data
%cd lab
!find data -type f | head

In [ ]:
# 1. Verifier tests (evidence saved to results/)
!mkdir -p results && python test_verifier.py | tee results/verifier_tests.txt

In [ ]:
# 2. Rollout calibration: 2x4 vs 1x8 on the 128 calibration cases
!python grpo_lab.py calib --data data

In [ ]:
# 3. 200-step GRPO run (group size = calibration recommendation; override with --group-size 4|8)
!python grpo_lab.py train --data data

In [ ]:
# 4. Base vs adapter on the 240 held-out cases
!python grpo_lab.py eval --data data

In [ ]:
# 5. Tables for the report
!python grpo_lab.py report && python make_figures.py
!pip freeze > results/pip_freeze.txt

In [ ]:
# 6. Package the submission (fill in REPORT.md first, or re-run this cell after editing it)
!rm -f ../submission.zip && zip -qr ../submission.zip README.md REPORT.md grpo_lab.py verifier.py test_verifier.py make_figures.py grpo_colab.ipynb lora_adapter results
files.download('../submission.zip')